In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 300
plt.rcParams["hatch.linewidth"] = 0.3
import pandas as pd
import stylia

stylia.set_style("ersilia")
nc = stylia.NamedColors()
import os

In [ ]:
datasets = [
    ["CHEMBL", "chembl4649948"],
    ["CHEMBL", "chembl4659961"],
    ["TDC", "ames"],
    ["TDC", "bbb_martins"],
    ["TDC", "bioavailability_ma"],
    ["TDC", "carcinogens_lagunin"],
    ["TDC", "clintox"],
    ["TDC", "cyp1a2_veith"],
    ["TDC", "cyp2c19_veith"],
    ["TDC", "cyp2c9_substrate_carbonmangels"],
    ["TDC", "cyp2c9_veith"],
    ["TDC", "cyp2d6_substrate_carbonmangels"],
    ["TDC", "cyp2d6_veith"],
    ["TDC", "cyp3a4_substrate_carbonmangels"],
    ["TDC", "cyp3a4_veith"],
    ["TDC", "dili"],
    ["TDC", "herg"],
    ["TDC", "hia_hou"],
    ["TDC", "pgp_broccatelli"],
    ["TDC", "skin_reaction"],
]

PATH_TO_EOSBENCH_RESULTS = os.path.join("../results/eosbench")

In [ ]:
# Create results df
files = sorted(os.listdir(PATH_TO_EOSBENCH_RESULTS))
df = pd.concat(
    [pd.read_csv(os.path.join(PATH_TO_EOSBENCH_RESULTS, f)) for f in files],
    ignore_index=True,
)
df["source"] = [i.upper() for i in df["source"]]

In [ ]:
stylia.set_format("print")

fig, axs = stylia.create_figure(4, 6, width=2, height=1)

for source, dataset in datasets:
    # Get values
    df_subset = df[(df["dataset"] == dataset) & (df["source"] == source)]

    # Prepare plot
    ax = axs.next()
    ax.set_title(f"Source: {source}\nDataset: {dataset}  -  Folds: {len(df_subset)}/5")
    ax.set_ylim([0, 1])
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.set_xlim([0, 6])
    w = 1
    x = [1, 2, 4, 5]
    ax.set_xticks(x)
    ax.set_xticklabels(["AUROC\n(base)", "AUROC\n(LQ)", "AUPR\n(base)", "AUPR\n(LQ)"])

    try:
        auroc_mean = df_subset["auroc"].mean()
        auroc_std = df_subset["auroc"].std()
        aupr_mean = df_subset["aupr"].mean()
        aupr_std = df_subset["aupr"].std()
        auroc_ref_mean = df_subset["auroc_ref_mean"].tolist()[0]
        auroc_ref_std = df_subset["auroc_ref_std"].tolist()[0]
        aupr_ref_mean = df_subset["aupr_ref_mean"].tolist()[0]
        aupr_ref_std = df_subset["aupr_ref_std"].tolist()[0]
        y1 = [auroc_ref_mean, auroc_mean, aupr_ref_mean, aupr_mean]
        y2 = [auroc_ref_std, auroc_std, aupr_ref_std, aupr_std]
        aupr_baseline_mean = df_subset["aupr_baseline"].mean()
        aupr_baseline_std = df_subset["aupr_baseline"].std()
        aupr_baseline_ref = (
            df_subset["n_pos_train"].tolist()[0] + df_subset["n_pos_test"].tolist()[0]
        ) / (df_subset["n_train"].tolist()[0] + df_subset["n_test"].tolist()[0])
        y_baseline = [0.5, 0.5, aupr_baseline_ref, aupr_baseline_mean]
        y_baseline_std = [0, 0, 0, aupr_baseline_std]
        colors = [nc.gray, nc.yellow, nc.gray, nc.blue]

        # Plot
        for i in range(len(x)):
            ax.bar(x[i], y1[i], color=colors[i], width=w, ec="k", lw=0.5)
            ax.vlines(x[i], y1[i] - y2[i], y1[i] + y2[i], color="k", lw=0.5)
            ax.bar(
                x[i],
                y_baseline[i],
                color=colors[i],
                width=w,
                ec="k",
                lw=0.5,
                hatch="/" * 6,
            )
            ax.vlines(
                x[i],
                y_baseline[i] - y_baseline_std[i],
                y_baseline[i] + y_baseline_std[i],
                color="k",
                lw=0.5,
            )

    except:
        pass

plt.tight_layout()
plt.show()

In [ ]:
df

In [ ]:
y2